2.多层感知机

![2.1理论计算题](2.1理论计算题.jpg)

In [ ]:

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # 解决MKL冲突
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.transforms as transforms
import gc  # 垃圾回收


# 1. 数据加载

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

batch_size = 64
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# 2. 参数初始化

input_dim = 784
hidden_dim = 256
output_dim = 10

W1 = torch.randn(input_dim, hidden_dim) * 0.01
b1 = torch.zeros(hidden_dim)
W2 = torch.randn(hidden_dim, output_dim) * 0.01
b2 = torch.zeros(output_dim)

W1.requires_grad_(True)
b1.requires_grad_(True)
W2.requires_grad_(True)
b2.requires_grad_(True)

# 3. 激活 & 损失

def relu(x):
    return torch.max(torch.tensor(0.0), x)

def softmax(x):
    max_vals, _ = torch.max(x, dim=1, keepdim=True)
    exp_x = torch.exp(x - max_vals)
    sum_exp_x = torch.sum(exp_x, dim=1, keepdim=True)
    return exp_x / sum_exp_x

def cross_entropy_loss(y_pred, y_true):
    batch_size = y_pred.shape[0]
    probs = softmax(y_pred)
    log_probs = -torch.log(probs[range(batch_size), y_true])
    return torch.mean(log_probs)


# 4. 训练

learning_rate = 0.1
epochs = 10

train_losses = []
train_accs = []

for epoch in range(epochs):
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        # 前向
        h1 = torch.matmul(images, W1) + b1
        h1_relu = relu(h1)
        out = torch.matmul(h1_relu, W2) + b2

        # 损失
        loss = cross_entropy_loss(out, labels)

        # 反向 + 更新
        loss.backward()
        with torch.no_grad():
            W1 -= learning_rate * W1.grad
            b1 -= learning_rate * b1.grad
            W2 -= learning_rate * W2.grad
            b2 -= learning_rate * b2.grad
            W1.grad.zero_()
            b1.grad.zero_()
            W2.grad.zero_()
            b2.grad.zero_()

        total_loss += loss.item() * images.shape[0]
        _, predicted = torch.max(out.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(train_dataset)
    avg_acc = 100 * correct / total
    train_losses.append(avg_loss)
    train_accs.append(avg_acc)
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.2f}%')

gc.collect()
torch.cuda.empty_cache()

# 5. 测试

correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        h1 = torch.matmul(images, W1) + b1
        h1_relu = relu(h1)
        out = torch.matmul(h1_relu, W2) + b2
        _, predicted = torch.max(out.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy: {100 * correct / total:.2f}%')

# 6. 绘图
# --------------------------
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(train_accs)
plt.title('Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')

plt.savefig("training_curve.png", dpi=300, bbox_inches='tight')
plt.close()  # 关键：立刻释放figure内存
print("曲线已保存为 training_curve.png")

Epoch [1/10], Loss: 0.6859, Acc: 75.78%
Epoch [2/10], Loss: 0.4499, Acc: 83.99%
Epoch [3/10], Loss: 0.4006, Acc: 85.64%
Epoch [4/10], Loss: 0.3728, Acc: 86.56%
Epoch [5/10], Loss: 0.3531, Acc: 87.22%
Epoch [6/10], Loss: 0.3364, Acc: 87.71%
Epoch [7/10], Loss: 0.3227, Acc: 88.17%
Epoch [8/10], Loss: 0.3114, Acc: 88.59%
Epoch [9/10], Loss: 0.3004, Acc: 88.98%
Epoch [10/10], Loss: 0.2923, Acc: 89.28%
Test Accuracy: 83.30%
曲线已保存为 training_curve.png


3.模型选择，权重衰减和丢弃法

第 1 题
训练误差是模型在训练数据集上得到的误差，体现模型对训练样本的拟合能力；
泛化误差则是模型在陌生的测试数据或真实场景数据上产生的误差，代表模型对未知数据的预测能力。
当训练误差很低但泛化误差偏高时，说明模型出现了过拟合，此时模型过度学习了训练数据里的噪声与局部特征，没能捕捉数据的通用规律。
想要缓解过拟合，可以采用：
一是简化模型结构，降低模型自身复杂度；
二是引入正则化，例如 L2 权重衰减、Dropout 等，约束参数更新或随机停用部分神经元；
三是扩充训练数据或使用数据增强，丰富样本多样性；此外也可以采用提前停止策略，在验证集误差开始上升时终止训练。


第 2 题
K 折交叉验证是常用的模型评估与超参数选择方法，具体流程如下：
首先将全部数据集随机划分为 K 份互不重叠、规模相近的子集；
随后依次开展 K 轮实验，每一轮都选取其中一折作为验证集，剩余 K-1 折作为训练集来训练模型，并计算对应验证集指标；
完成所有轮次后，将 K 组验证结果取平均值，以此作为模型泛化能力的最终评价依据。
借助该方法可以客观评估模型性能，也能根据验证结果筛选出最优超参数，最后再使用完整训练集训练得到最终模型。

In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# 1. 数据加载与预处理（Fashion-MNIST）

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  # 展平为 784 维向量
])

# 加载数据集
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

# 划分训练集为训练/验证（8:2）
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset, [train_size, val_size]
)

batch_size = 64
train_loader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False)

# 2. 自定义组件实现

# 2.1 带 L2 权重衰减的 SGD 优化器
class SGDWithWeightDecay:
    def __init__(self, params, lr, weight_decay):
        self.params = list(params)
        self.lr = lr
        self.weight_decay = weight_decay  # 正则化系数 λ

    def step(self):
        for param in self.params:
            if param.grad is not None:
                # 题目要求：旧权重先乘以 (1 - ηλ)
                param.data = param.data * (1 - self.lr * self.weight_decay)
                # 再执行梯度更新
                param.data -= self.lr * param.grad

    def zero_grad(self):
        for param in self.params:
            if param.grad is not None:
                param.grad.zero_()

# 2.2 Dropout 层从零实现
def dropout_layer(X, dropout, is_training):
    """
    实现 Dropout：训练时随机置0并缩放，测试时直接返回
    :param X: 输入张量
    :param dropout: 丢弃概率（0~1）
    :param is_training: 是否为训练模式
    """
    if not is_training or dropout == 0:
        return X
    # 随机生成掩码（保留概率 1-dropout）
    mask = (torch.rand(X.shape) > dropout).float()
    # 缩放输出，保证训练/测试期望一致
    return mask * X / (1 - dropout)

# 2.3 ReLU 激活函数
def relu(x):
    return torch.max(torch.tensor(0.0), x)

# 2.4 Softmax 交叉熵损失
def cross_entropy_loss(y_pred, y_true):
    batch_size = y_pred.shape[0]
    # 数值稳定的 Softmax
    max_vals, _ = torch.max(y_pred, dim=1, keepdim=True)
    exp_x = torch.exp(y_pred - max_vals)
    sum_exp_x = torch.sum(exp_x, dim=1, keepdim=True)
    probs = exp_x / sum_exp_x
    # 交叉熵计算
    log_probs = -torch.log(probs[range(batch_size), y_true])
    return torch.mean(log_probs)

# 3. 三种模型配置对比

input_dim = 784    # 28*28
hidden_dim = 256   # 隐藏层神经元数（较大易过拟合）
output_dim = 10    # 10 分类
lr = 0.1
epochs = 15        # 训练轮数

# 三种配置
configs = [
    {"name": "无正则化", "weight_decay": 0, "dropout": 0},
    {"name": "L2 权重衰减", "weight_decay": 1e-4, "dropout": 0},
    {"name": "Dropout", "weight_decay": 0, "dropout": 0.2}
]

results = []

# 4. 训练循环

for config in configs:
    # 初始化参数
    W1 = torch.randn(input_dim, hidden_dim) * 0.01
    b1 = torch.zeros(hidden_dim)
    W2 = torch.randn(hidden_dim, output_dim) * 0.01
    b2 = torch.zeros(output_dim)
    params = [W1, b1, W2, b2]
    for p in params:
        p.requires_grad_(True)

    # 初始化优化器（带权重衰减）
    optimizer = SGDWithWeightDecay(params, lr, config["weight_decay"])

    train_loss_history = []
    val_loss_history = []

    for epoch in range(epochs):
        # 训练阶段
        total_train_loss = 0.0
        for images, labels in train_loader:
            batch_size = images.shape[0]
            optimizer.zero_grad()

            # 前向传播（带 Dropout）
            h1 = torch.matmul(images, W1) + b1
            h1_relu = relu(h1)
            h1_drop = dropout_layer(h1_relu, config["dropout"], is_training=True)
            out = torch.matmul(h1_drop, W2) + b2

            # 损失计算与反向传播
            loss = cross_entropy_loss(out, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * batch_size

        avg_train_loss = total_train_loss / len(train_subset)
        train_loss_history.append(avg_train_loss)

        # 验证阶段（Dropout 关闭）
        with torch.no_grad():
            total_val_loss = 0.0
            for images, labels in val_loader:
                h1 = torch.matmul(images, W1) + b1
                h1_relu = relu(h1)
                out = torch.matmul(h1_relu, W2) + b2
                val_loss = cross_entropy_loss(out, labels)
                total_val_loss += val_loss.item() * images.shape[0]

            avg_val_loss = total_val_loss / len(val_subset)
            val_loss_history.append(avg_val_loss)

        print(f'[{config["name"]}] Epoch {epoch+1:2d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

    results.append({
        "name": config["name"],
        "train_loss": train_loss_history,
        "val_loss": val_loss_history
    })


# 5. 绘制对比 Loss 曲线

plt.figure(figsize=(12, 5))

# 训练损失曲线
plt.subplot(1, 2, 1)
for res in results:
    plt.plot(range(1, epochs+1), res["train_loss"], label=res["name"])
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

# 验证损失曲线
plt.subplot(1, 2, 2)
for res in results:
    plt.plot(range(1, epochs+1), res["val_loss"], label=res["name"])
plt.title("Validation Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

[无正则化] Epoch  1 | Train Loss: 0.7262 | Val Loss: 0.5311
[无正则化] Epoch  2 | Train Loss: 0.4614 | Val Loss: 0.4296
[无正则化] Epoch  3 | Train Loss: 0.4101 | Val Loss: 0.4077
[无正则化] Epoch  4 | Train Loss: 0.3826 | Val Loss: 0.3930
[无正则化] Epoch  5 | Train Loss: 0.3622 | Val Loss: 0.3758
[无正则化] Epoch  6 | Train Loss: 0.3441 | Val Loss: 0.3651
[无正则化] Epoch  7 | Train Loss: 0.3323 | Val Loss: 0.3609
[无正则化] Epoch  8 | Train Loss: 0.3212 | Val Loss: 0.3661
[无正则化] Epoch  9 | Train Loss: 0.3106 | Val Loss: 0.3472
[无正则化] Epoch 10 | Train Loss: 0.3011 | Val Loss: 0.3622
[无正则化] Epoch 11 | Train Loss: 0.2930 | Val Loss: 0.3332
[无正则化] Epoch 12 | Train Loss: 0.2863 | Val Loss: 0.3349
[无正则化] Epoch 13 | Train Loss: 0.2798 | Val Loss: 0.3321
[无正则化] Epoch 14 | Train Loss: 0.2703 | Val Loss: 0.3295
[无正则化] Epoch 15 | Train Loss: 0.2648 | Val Loss: 0.3240
[L2 权重衰减] Epoch  1 | Train Loss: 0.7316 | Val Loss: 0.5188
[L2 权重衰减] Epoch  2 | Train Loss: 0.4678 | Val Loss: 0.4927
[L2 权重衰减] Epoch  3 | Train Loss: 0.4158 | 

C:\Users\86181\AppData\Local\Temp\ipykernel_38612\2860361198.py:190: UserWarning: Glyph 26080 (\N{CJK UNIFIED IDEOGRAPH-65E0}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\2860361198.py:190: UserWarning: Glyph 27491 (\N{CJK UNIFIED IDEOGRAPH-6B63}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\2860361198.py:190: UserWarning: Glyph 21017 (\N{CJK UNIFIED IDEOGRAPH-5219}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\2860361198.py:190: UserWarning: Glyph 21270 (\N{CJK UNIFIED IDEOGRAPH-5316}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\2860361198.py:190: UserWarning: Glyph 26435 (\N{CJK UNIFIED IDEOGRAPH-6743}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\2860361198.py:190: UserWarning: Glyph 37

4.数值稳定性和激活函数

(1)梯度消失与梯度爆炸的量化分析

在 d 层深层神经网络的反向传播中，梯度计算包含多层矩阵与激活函数导数的连乘项。
当权重矩阵的谱范数与激活函数导数的乘积恒大于 1时，梯度会随网络层数增加呈指数级放大，导致梯度爆炸，最终因数值溢出而无法正常更新参数；
当权重矩阵的谱范数与激活函数导数的乘积恒小于 1时，梯度会随网络层数增加呈指数级衰减，导致梯度消失，靠近输入层的参数无法获得有效更新，模型难以收敛。
传统 sigmoid、tanh 等饱和激活函数在输入远离 0 点时导数趋近于 0，会加剧梯度消失问题；而权重初始化不当（如权重过大）则易引发梯度爆炸。

(2)ReLU 缓解梯度消失问题的原因

ReLU 激活函数定义为 ReLU(x)=max(0,x)，其核心优势在于：
当输入 x>0时，导数恒为 1，不存在梯度趋近于 0 的饱和区间，能让梯度在反向传播中无衰减地传递，避免了多层连乘导致的梯度指数级衰减；
当输入 x<0时导数为 0，仅会使部分神经元失活，不会影响整体梯度的传递。因此，ReLU 能从根本上缓解传统饱和激活函数带来的梯度消失问题，提升深层网络的训练稳定性。

In [4]:
import torch
import torch.nn as nn
import torch.nn.init as init
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 1. 工具函数：打印各层梯度范数

def print_grad_norms(model, name):
    print(f"\n--- {name} 各层梯度范数 ---")
    norms = []
    for i, layer in enumerate(model):
        if isinstance(layer, nn.Linear):
            if layer.weight.grad is not None:
                norm = torch.norm(layer.weight.grad, 2).item()
                norms.append(norm)
                print(f"层{i:2d} 梯度范数: {norm:.6e}")
            else:
                norms.append(0.0)
                print(f"层{i:2d} 梯度范数: None")
    return norms

# 2. 构建 20 层全连接网络（隐藏层宽度 256）

def build_deep_net(activation, init_fn=None, std=None):
    layers = []
    input_dim = 256  # 输入维度
    hidden_dim = 256 # 隐藏层宽度
    for i in range(20):
        layers.append(nn.Linear(input_dim, hidden_dim))
        if activation is not None:
            layers.append(activation())
        input_dim = hidden_dim
    model = nn.Sequential(*layers)
    
    # 权重初始化
    if init_fn is not None:
        for m in model:
            if isinstance(m, nn.Linear):
                if std is not None:
                    init_fn(m.weight, mean=0, std=std)
                else:
                    init_fn(m.weight)
                if m.bias is not None:
                    init.zeros_(m.bias)
    return model

# 3. 模拟梯度消失（Sigmoid + 普通高斯初始化 std=1）

print("===== 实验1：Sigmoid + 普通高斯初始化（模拟梯度消失） =====")
model1 = build_deep_net(nn.Sigmoid, init.normal_, std=1)
x = torch.randn(32, 256)  # 随机输入数据
y = torch.randint(0, 10, (32,))

model1.zero_grad()
out = model1(x)
loss = nn.CrossEntropyLoss()(out, y)
loss.backward()
norms1 = print_grad_norms(model1, "Sigmoid + 普通高斯初始化")


# 4. 模拟梯度爆炸（ReLU + 大初始值 std=10）

print("\n===== 实验2：ReLU + 大初始值 std=10（模拟梯度爆炸/NaN） =====")
model2 = build_deep_net(nn.ReLU, init.normal_, std=10)
x = torch.randn(32, 256)
y = torch.randint(0, 10, (32,))

model2.zero_grad()
out = model2(x)
print("模型输出是否有 NaN:", torch.isnan(out).any().item())

loss = nn.CrossEntropyLoss()(out, y)
loss.backward()
norms2 = print_grad_norms(model2, "ReLU + std=10 初始化")


# 5. 修复：Xavier 初始化 + ReLU（梯度稳定）

print("\n===== 实验3：Xavier 初始化 + ReLU（梯度稳定） =====")
model3 = build_deep_net(nn.ReLU, init.xavier_uniform_)
x = torch.randn(32, 256)
y = torch.randint(0, 10, (32,))

model3.zero_grad()
out = model3(x)
loss = nn.CrossEntropyLoss()(out, y)
loss.backward()
norms3 = print_grad_norms(model3, "Xavier 初始化 + ReLU")

# 6. 绘制三种初始化的梯度范数对比曲线

plt.figure(figsize=(10, 6))
layers = list(range(0, 40, 2))  # 仅取 Linear 层（共20层）

plt.plot(layers, norms1, marker='o', label='Sigmoid + Normal(std=1)', color='red')
plt.plot(layers, norms2, marker='s', label='ReLU + Normal(std=10)', color='orange')
plt.plot(layers, norms3, marker='^', label='ReLU + Xavier', color='green')

plt.yscale('log')  # 对数坐标，方便观察差异
plt.xlabel('Layer Index')
plt.ylabel('Gradient Norm (log scale)')
plt.title('不同初始化策略的梯度范数对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("init_gradient_norm.png", dpi=300, bbox_inches='tight')
print("\n梯度范数对比图已保存为: init_gradient_norm.png")

===== 实验1：Sigmoid + 普通高斯初始化（模拟梯度消失） =====

--- Sigmoid + 普通高斯初始化 各层梯度范数 ---
层 0 梯度范数: 1.086410e+01
层 2 梯度范数: 7.678189e+00
层 4 梯度范数: 6.480395e+00
层 6 梯度范数: 5.890894e+00
层 8 梯度范数: 4.465477e+00
层10 梯度范数: 4.148734e+00
层12 梯度范数: 3.060926e+00
层14 梯度范数: 2.498677e+00
层16 梯度范数: 2.226661e+00
层18 梯度范数: 1.447600e+00
层20 梯度范数: 1.352615e+00
层22 梯度范数: 1.101136e+00
层24 梯度范数: 9.377560e-01
层26 梯度范数: 7.211717e-01
层28 梯度范数: 6.026247e-01
层30 梯度范数: 4.008044e-01
层32 梯度范数: 3.405751e-01
层34 梯度范数: 3.517164e-01
层36 梯度范数: 2.697048e-01
层38 梯度范数: 2.621817e-01

===== 实验2：ReLU + 大初始值 std=10（模拟梯度爆炸/NaN） =====
模型输出是否有 NaN: True

--- ReLU + std=10 初始化 各层梯度范数 ---
层 0 梯度范数: nan
层 2 梯度范数: nan
层 4 梯度范数: nan
层 6 梯度范数: nan
层 8 梯度范数: nan
层10 梯度范数: nan
层12 梯度范数: nan
层14 梯度范数: nan
层16 梯度范数: nan
层18 梯度范数: nan
层20 梯度范数: nan
层22 梯度范数: nan
层24 梯度范数: nan
层26 梯度范数: nan
层28 梯度范数: nan
层30 梯度范数: nan
层32 梯度范数: nan
层34 梯度范数: nan
层36 梯度范数: nan
层38 梯度范数: nan

===== 实验3：Xavier 初始化 + ReLU（梯度稳定） =====

--- Xavier 初始化 + ReLU 各层梯度范数 ---
层 0 梯度范数:

C:\Users\86181\AppData\Local\Temp\ipykernel_38612\18804124.py:107: UserWarning: Glyph 19981 (\N{CJK UNIFIED IDEOGRAPH-4E0D}) missing from font(s) DejaVu Sans.
  plt.savefig("init_gradient_norm.png", dpi=300, bbox_inches='tight')
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\18804124.py:107: UserWarning: Glyph 21516 (\N{CJK UNIFIED IDEOGRAPH-540C}) missing from font(s) DejaVu Sans.
  plt.savefig("init_gradient_norm.png", dpi=300, bbox_inches='tight')
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\18804124.py:107: UserWarning: Glyph 21021 (\N{CJK UNIFIED IDEOGRAPH-521D}) missing from font(s) DejaVu Sans.
  plt.savefig("init_gradient_norm.png", dpi=300, bbox_inches='tight')
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\18804124.py:107: UserWarning: Glyph 22987 (\N{CJK UNIFIED IDEOGRAPH-59CB}) missing from font(s) DejaVu Sans.
  plt.savefig("init_gradient_norm.png", dpi=300, bbox_inches='tight')
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\18804124.py:107: UserWarning: Glyp


梯度范数对比图已保存为: init_gradient_norm.png


5.泛化表现，协变量偏移和对抗性数据

（1）协变量偏移（Covariate Shift）
协变量偏移是指训练数据与测试数据的输入分布不同（p(x)不等于q(x)），但输入到标签的条件分布保持不变（p(y∣x)=q(y∣x)）。
以电商推荐系统为例：模型在双 11 的用户数据上训练，此时高价商品的曝光和购买占比高，训练输入分布偏向高价商品；但在日常场景中，用户更多浏览低价商品，测试输入分布发生了明显变化。然而，用户 “对感兴趣的商品会点击” 的行为逻辑（即 p(y∣x)）并未改变。这种偏移会导致模型对新输入分布的样本泛化能力下降，可通过对训练样本按测试分布加权或重采样进行修正。
（2）标签偏移（Label Shift）
标签偏移是指训练数据与测试数据的标签分布不同（p(y)不等于q(y)），但标签到输入的条件分布保持不变（p(x∣y)=q(x∣y)）。以医疗肺炎诊断模型为例：模型在三甲医院的数据上训练，重症肺炎患者占比高，标签分布偏向重症；但部署到社区医院时，接诊的多为轻症患者，重症标签占比大幅降低。然而，重症肺炎患者的 X 光影像特征（即 重症肺炎）本身并未改变。这种偏移会导致模型对标签占比低的类别预测偏差增大，可通过按测试集标签占比调整损失权重来缓解。
（3）两者的区别与联系
区别：协变量偏移的核心是输入分布变化，标签条件分布不变；标签偏移的核心是标签分布变化，输入条件分布不变，二者分布变化的对象不同，对模型的影响和修正思路也不同。
联系：二者都属于分布偏移的典型类型，均因训练与测试数据不满足独立同分布假设，导致模型泛化能力下降，都需要通过领域自适应、重加权等方法进行缓解。

In [5]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error


# 1. 人工数据集构造（训练集P）

np.random.seed(42)  # 固定随机种子保证可复现
n_train = 1000
# 训练集P: x ~ N(-1, 1), y = 2x + ε
x_train = np.random.normal(loc=-1, scale=1, size=(n_train, 1))
epsilon_train = np.random.normal(loc=0, scale=0.1, size=(n_train, 1))
y_train = 2 * x_train + epsilon_train


# 2. 测试集Q构造（协变量偏移）

n_test = 500
# 测试集Q: x ~ N(2, 1)，与训练集分布差异明显
x_test = np.random.normal(loc=2, scale=1, size=(n_test, 1))
epsilon_test = np.random.normal(loc=0, scale=0.1, size=(n_test, 1))
y_test = 2 * x_test + epsilon_test

# 3. 基线模型：普通线性回归

baseline_model = LinearRegression()
baseline_model.fit(x_train, y_train)
y_pred_baseline = baseline_model.predict(x_test)
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
print(f"基线模型测试MSE: {mse_baseline:.4f}")


# 4. 偏移校正：逻辑回归分类器计算权重

# 4a 混合数据并标记（训练集=0，测试集=1）
x_all = np.vstack([x_train, x_test])
domain_labels = np.hstack([np.zeros(n_train), np.ones(n_test)])  # 训练集0，测试集1

# 训练分类器预测样本属于测试集的概率
domain_clf = LogisticRegression(random_state=42)
domain_clf.fit(x_all, domain_labels)
p_test_given_x = domain_clf.predict_proba(x_train)[:, 1]  # P(test|x)

# 4b 计算训练样本权重 w_i ∝ P(test|x_i)/P(train|x_i)
p_train_given_x = 1 - p_test_given_x
weights = p_test_given_x / p_train_given_x
weights = weights / weights.mean()  # 权重归一化，避免数值过大


# 5. 加权线性回归模型

weighted_model = LinearRegression()
weighted_model.fit(x_train, y_train, sample_weight=weights)
y_pred_weighted = weighted_model.predict(x_test)
mse_weighted = mean_squared_error(y_test, y_pred_weighted)
print(f"加权修正后测试MSE: {mse_weighted:.4f}")

# 6. 可视化对比

plt.figure(figsize=(10, 5))

# 数据分布对比
plt.subplot(1, 2, 1)
plt.scatter(x_train, y_train, alpha=0.5, label='训练集P (x~N(-1,1))')
plt.scatter(x_test, y_test, alpha=0.5, label='测试集Q (x~N(2,1))')
plt.plot(x_test, y_pred_baseline, 'r--', label='基线模型预测')
plt.plot(x_test, y_pred_weighted, 'g--', label='加权模型预测')
plt.xlabel('x')
plt.ylabel('y')
plt.title('数据分布与模型预测')
plt.legend()

# MSE对比
plt.subplot(1, 2, 2)
plt.bar(['基线模型', '加权修正模型'], [mse_baseline, mse_weighted], color=['red', 'green'])
plt.ylabel('测试MSE')
plt.title('校正前后性能对比')

plt.tight_layout()
plt.savefig("covariate_shift_correction.png", dpi=300, bbox_inches='tight')
plt.close()
print("对比图已保存为 covariate_shift_correction.png")

基线模型测试MSE: 0.0102
加权修正后测试MSE: 0.0240


C:\Users\86181\AppData\Local\Temp\ipykernel_38612\573824080.py:82: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\573824080.py:82: UserWarning: Glyph 25454 (\N{CJK UNIFIED IDEOGRAPH-636E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\573824080.py:82: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\573824080.py:82: UserWarning: Glyph 24067 (\N{CJK UNIFIED IDEOGRAPH-5E03}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\573824080.py:82: UserWarning: Glyph 19982 (\N{CJK UNIFIED IDEOGRAPH-4E0E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86181\AppData\Local\Temp\ipykernel_38612\573824080.py:82: UserWarning: Glyph 27169 (\N{CJK 

对比图已保存为 covariate_shift_correction.png
